In [1]:
## Imports

import sys, os
from pathlib import Path

parent_folder = str(Path.cwd().parents[1])
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

from sigpy import mri
import scipy
import pickle
from sklearn.decomposition import PCA
import seaborn as sns
import sigpy as sp
import cupy as cp
import numpy as np
from scipy.io import savemat
import twixtools
import matplotlib.pyplot as plt
import pickle_utils
import mapvbvd

### 1. Load data

In [2]:
data_file = '/data/lilianae/NaF_Patient3/twix/anon_meas_MID00283_FID65311_Tho_fl3d_star_vibe_991_nav_tj_2000sp_AllCoils_SOS.dat'

multi_twix = twixtools.read_twix(str(data_file))

Software version: VD/VE (!?)

Scan  0


100%|██████████| 15.9G/15.9G [00:13<00:00, 1.27GB/s]


### 2. Extract MDBs from Twix file

In [3]:
def extract_image_mdbs(multi_twix):
    '''
    Extract Measurement Data Blocks (mdbs) that pertain to image data.

    Input
    ----------------------------------
    multi_twix: list of twix objects
        Result of read_twix(). multi_twix[-1] contains the following 
        attributes (accessed like ['attribute']): hdr, hdr_str, mdb, geometry

    Output
    ----------------------------------
    mdb_list_for_image: list of twix mdbs
        List of MDBs containing image information

    '''
    image_mdbs = []

    for i, mdb in enumerate(multi_twix[-1]['mdb']):
    ## Use same logic as twix_category['image'] to get mdh values for k-space
        if (not mdb.is_flag_set('SYNCDATA') and
            not mdb.is_flag_set('ACQEND') and
            not mdb.is_flag_set('RTFEEDBACK') and
            not mdb.is_flag_set('HPFEEDBACK') and
            not mdb.is_flag_set('REFPHASESTABSCAN') and
            not mdb.is_flag_set('PHASESTABSCAN') and
            not mdb.is_flag_set('PHASCOR') and
            not mdb.is_flag_set('NOISEADJSCAN') and
            not mdb.is_flag_set('noname60') and
            (not mdb.is_flag_set('PATREFSCAN') or mdb.is_flag_set('PATREFANDIMASCAN'))):

            if not np.isnan(mdb.mdh.TimeStamp):
                ## Extract k-space data for this readout
                mdb_data = mdb.data  # Shape: (channels, samples)
                image_mdbs.append(mdb)

    return image_mdbs          

In [4]:
image_mdbs = extract_image_mdbs(multi_twix)
print(f'len(image_mdbs) = {len(image_mdbs)}')
print(f'Example: image_mdbs[0].data.shape = {image_mdbs[0].data.shape}')

len(image_mdbs) = 232232
Example: image_mdbs[0].data.shape = (15, 704)


### 3. Create k-space array from MDBs

In [ ]:
def sort_mdbs_to_kspace(image_mdbs):
    n_cha, max_col = image_mdbs[0].data.shape

    n_echo = 1 + max([mdb.mdh.Counter.Eco for mdb in image_mdbs])
    n_line = 1 + max([mdb.cLin for mdb in image_mdbs])
    n_part = 1 + max([mdb.cPar for mdb in image_mdbs])

    ## Initialize array (Part, Line, Echo, Channel, Col)
    out = np.zeros([n_part, n_line, n_echo, n_cha, max_col], dtype=np.complex64)

    for mdb in image_mdbs:
        e = mdb.mdh.Counter.Eco        ## Get the Echo index (0 or 1)
        p = mdb.cPar                   ## Get Partition index
        l = mdb.cLin                   ## Get Spoke index
        
        # Get raw data (could be 512 or 704, 704=first echo, 512=second echo)
        raw_data = mdb.data 
        curr_len = raw_data.shape[-1]
        
        ## Put data into the array: 
        ## If curr_len < max_col, the rest remains zeros (like mapVBVD MATLAB)
        out[p, l, e, :, :curr_len] += raw_data

    return out

Reshape array to:
 1. Get only first echo (this contains actual image measurement, we do this in MATLAB) 
 2. Follow dimension conventions

In [ ]:
kspace_array_2_echoes_py = sort_mdbs_to_kspace(image_mdbs=image_mdbs)
print(f'kspace_array_2_echoes_py.shape = {kspace_array_2_echoes_py.shape}')
kspace_array_echo1_py = np.transpose(kspace_array_2_echoes_py[:, :, 0, :, :], (2, 0, 1, 3))    ## Select first echo (contains actual image data, as we do in MATLAB)
print(f'kspace_array_echo1_py.shape = {kspace_array_echo1_py.shape}')                          ## Reshape to convention: (ncoils, nslices, nspokes, nreadouts)

kspace_array_2_echoes_py.shape = (58, 2002, 2, 15, 704)
kspace_array_echo1_py.shape = (15, 58, 2002, 704)


### 4. Follow same DC extraction logic as MATLAB

In [ ]:
ncoils, nslices, nspokes, nro = kspace_array_echo1_py.shape

ksp_resp_list_py = []
for c in range(ncoils):
    ksp_tmp_py = kspace_array_echo1_py[c,...] 
    ksp_resp_coil = ksp_tmp_py[19:26, :, 247:267]

    ksp_resp_list_py.append(ksp_resp_coil)

ksp_resp_py = np.stack(ksp_resp_list_py, axis=0)
print(f'ksp_resp_py = {ksp_resp_py.shape}')

ksp_resp_py = (15, 7, 2002, 20)


### 5. Load k-space array from MATLAB (DC extracted, lines 12-21)

In [14]:
from scipy.io import loadmat

data = loadmat('/home/lilianae/projects/naf_clean/load_data_pipeline/comparison_subject3/resp_signal_all_vars_subject3_mid0283.mat')

ksp_tmp_mat = data['ksp_tmp'].squeeze()
ksp_resp_init_mat = data['ksp_resp']

ksp_resp_mat = np.transpose(ksp_resp_init_mat, (1, 3, 2, 0)) 

print(f'PYTHON')
print("=" * 20)
print(f'ksp_resp_py.shape = {ksp_resp_py.shape}')

print(f'MATLAB')
print("=" * 20)
print(f'ksp_resp_mat.shape = {ksp_resp_mat.shape}')

PYTHON
ksp_resp_py.shape = (15, 7, 2002, 20)
MATLAB
ksp_resp_mat.shape = (15, 7, 2002, 20)


### 6. Compare Python vs MATLAB kspace arrays

In [ ]:
def compare_table(py_arr, mat_arr, title, coil_idx, partition_idx=0, spoke_idx=0):
    py_slice  = py_arr[coil_idx, partition_idx, spoke_idx, :]
    mat_slice = mat_arr[coil_idx, partition_idx, spoke_idx, :]

    def fmt_complex(c):
        return f"({c.real:.4e}{'+' if c.imag >= 0 else ''}{c.imag:.4e}j)"

    print(f"\n{title}")
    print(f"{'idx':<5} {'Python':^32} {'MATLAB':^32}")
    print("-" * 72)
    for i, (p, m) in enumerate(zip(py_slice, mat_slice)):
        print(f"{i:<5} {fmt_complex(p):^32} {fmt_complex(m):^32}")

compare_table(ksp_resp_py, ksp_resp_mat, title="Coil 0", coil_idx=0)
compare_table(ksp_resp_py, ksp_resp_mat, title="Coil 5", coil_idx=5)
compare_table(ksp_resp_py, ksp_resp_mat, title="Coil 0, Partition 2", coil_idx=0, partition_idx=2)


Coil 0
idx                Python                           MATLAB             
------------------------------------------------------------------------
0         (1.3227e-04+3.0445e-04j)         (1.3227e-04+3.0445e-04j)    
1        (-1.8037e-04+5.8314e-05j)        (-1.8037e-04+5.8314e-05j)    
2        (-2.5378e-04-3.6557e-04j)        (-2.5378e-04-3.6557e-04j)    
3         (1.2917e-04-4.2759e-04j)         (1.2917e-04-4.2759e-04j)    
4         (6.2978e-04+7.8299e-05j)         (6.2978e-04+7.8299e-05j)    
5         (7.4173e-04+7.3275e-04j)         (7.4173e-04+7.3275e-04j)    
6         (2.8204e-04+9.5756e-04j)         (2.8204e-04+9.5756e-04j)    
7        (-4.3564e-04+5.5214e-04j)        (-4.3564e-04+5.5214e-04j)    
8        (-8.2411e-04-1.4879e-04j)        (-8.2411e-04-1.4879e-04j)    
9        (-6.1580e-04-4.9997e-04j)        (-6.1580e-04-4.9997e-04j)    
10       (-1.9609e-04-2.0051e-04j)        (-1.9609e-04-2.0051e-04j)    
11       (-1.3995e-04+2.9013e-04j)        (-1.3995e-04+

In [27]:
# Check every possible coil mapping
for py_coil in range(15):
    for mat_coil in range(15):
        if np.allclose(ksp_resp_py[py_coil, :, :, :], ksp_resp_mat[mat_coil, :, :, :]):
            print(f"Python coil {py_coil} matches MATLAB coil {mat_coil}")

Python coil 0 matches MATLAB coil 0
